# HAM10000 · v6 · v5 + three switchable boosters (mixup/CutMix, colour constancy, 8-way TTA)

Same protocol as v4/v5 (lesion-grouped splits, duplicates removed from train only, per-epoch over/under-sampling, residual class weights, cosine + EMA, label smoothing, no metadata, validation-only selection, test evaluated once). All backbones from v5 still work (`convnext_tiny`, `efficientnet_b3`, `resnet50`, `swin_t`, `transnet`). **Tags start with `v6_`.**

**Honest framing:** nobody can promise these beat the current best (ConvNeXt-Tiny seed 42: test macro-F1 0.778). The test set is small (10 df, 16 vasc, 41 akiec images), so a single run differs from another by about +-0.03 by luck alone. What this notebook gives you is a fair, one-switch-at-a-time experiment.

| switch | what | why it might help | risk |
|---|---|---|---|
| `MIX_PROB`, `MIXUP_ALPHA`, `CUTMIX_ALPHA` | with probability `MIX_PROB` a batch is mixed (mixup or CutMix, 50/50) and the loss is the matching mix of two weighted losses | regularises; your logs show training loss far below validation loss | usually needs more epochs to pay off; may lower rare-class recall |
| `COLOR_CONSTANCY` | Shades-of-Gray illumination normalisation applied once to every image (train, val and test alike) | HAM10000 photos differ a lot in lighting/colour cast | changes the colours the pretrained network saw; needs `PRELOAD = True` |
| `TTA_MODE = "d4"` | predictions averaged over 8 flips/rotations instead of 4 flips | skin lesions have no fixed orientation; costs only inference time | tiny |

**How to run it (experiment plan)**
1. Dry run (`DRY_RUN = True`) once.
2. Real run A: `BACKBONE = "convnext_tiny"`, `SEED = 42`, defaults (mixup/CutMix on, d4 TTA on, colour constancy off). Compare **validation** macro-F1 and log-loss with the v5 ConvNeXt run (val F1 0.733 during training, log-loss 0.558 on the saved probabilities).
3. Real run B: add `COLOR_CONSTANCY = True`. Compare the same way.
4. Keep whatever helps on **validation**; run its seeds 43, 44; then ensemble with `HAM10000_final_ensemble.ipynb` (set `INCLUDE` to include `"v6_"`).

If the validation numbers get worse, set that switch off. Do not judge by the test column of a single seed.

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, time, random, glob, re, copy, math
import numpy as np
import pandas as pd

# ================= CONFIG: the only cell you edit between runs =================
BACKBONE     = "convnext_tiny"        # REFERENCE (non-CNN): "swin_t" | "transnet"   CNN: "efficientnet_b3" | "convnext_tiny" | "resnet50"
MEMBER       = "nikhil"        # who runs this (stored in the results file; NOT part of the tag)
WEIGHT_MODE  = "inverse"       # "none" | "inverse" | "sqrt" | "effective"  (computed from per-epoch counts if RESAMPLE)
DRY_RUN      = True            # True = tiny smoke run to check the pipeline. Set False for the real run.
SEED         = 42
EPOCHS       = None            # None = backbone preset (see PRESETS below); a number overrides it
BATCH_SIZE   = None
LR           = None
WEIGHT_DECAY = None
EFF_BETA     = 0.999           # only used by "effective"
IMG_SIZE     = None            # None = preset: efficientnet_b3 300 | convnext_tiny 224 | resnet50 224
COPY_LOCAL   = True            # copy images from Drive to Colab disk first

# ---- v4 training upgrades (each can be switched off to see what it adds) ----
USE_SCHEDULE  = True           # warm-up + cosine learning-rate decay
WARMUP_EPOCHS = 1
USE_EMA       = True           # evaluate a smoothed (moving-average) copy of the weights
EMA_DECAY     = 0.998
LABEL_SMOOTH  = 0.1            # 0 = off
AUG_LEVEL     = "strong"       # "base" (v3 augmentation) | "strong"
GRAD_CLIP     = None           # None = preset (2.0; transnet 1.0)
LR_NEW        = 5e-4           # transnet only: learning rate of the new (non-ResNet) layers
PRELOAD       = True           # decode + resize all images once into RAM (~2.7 GB at 300px). Auto-off if free RAM < 5 GB
RESUME        = True           # continue from ckpt_{tag}.pt in OUT_DIR if it exists (after a Colab disconnect)
SELECT        = "best"         # "best" = epoch with best validation macro-F1 | "last" = final epoch
USE_TTA       = True           # average predictions over flips/rotations in the final evaluation
TTA_MODE      = "d4"           # "flip" = 4 flips (v4/v5) | "d4" = 8 flips+rotations

# ---- v6 boosters (each can be switched off; see the notes at the top) ----
MIX_PROB      = 0.5            # share of training batches that are mixed (0 = off)
MIXUP_ALPHA   = 0.2            # mixup strength (0 = no mixup)
CUTMIX_ALPHA  = 1.0            # CutMix strength (0 = no CutMix)
COLOR_CONSTANCY = False        # Shades-of-Gray colour normalisation of every image (needs PRELOAD)
POSTHOC       = False          # v4 showed these offsets overfit validation; keep False for the backbone comparison
MEL_RECALL_FLOOR = None        # e.g. 0.70: post-hoc tuning may not push validation melanoma recall below this

# ---- resampling (TRAINING set only) ----
RESAMPLE            = True     # False = plain shuffle, weights from raw counts
TARGET_MODE         = "auto"   # "auto" (CAP_N / FLOOR_N / MAX_REPEAT) or "manual" (MANUAL_TARGETS)
CAP_N               = 1200     # class bigger than this  -> undersampled to this many images per epoch
FLOOR_N             = 500      # class smaller than this -> oversampled up to this many per epoch ...
MAX_REPEAT          = 6        # ... but an image is repeated at most this many times on average
MANUAL_TARGETS      = {"akiec": 500, "bcc": 500, "bkl": 880, "df": 500, "mel": 890, "nv": 1200, "vasc": 500}
RESAMPLE_EACH_EPOCH = True     # True = new random subset/repeats every epoch
LESION_AWARE        = True     # undersampling prefers images from lesions with few photos

# ---- near-duplicates (dHash, 256 bit). Only TRAIN images are ever removed. ----
DEDUP    = True
T_DUP    = 12                  # distance <= T_DUP  -> same picture (auto-removed from train)
T_REVIEW = 24                  # T_DUP < distance <= T_REVIEW -> reported only

DATA_DIR  = "/content/drive/MyDrive/HAM10000"                 # images + metadata (friend's shared folder)
SPLIT_DIR = DATA_DIR                                           # where train/val/test_split.csv live
OUT_DIR   = "/content/drive/MyDrive/HAM10000_model/imbalance_runs"
# ===============================================================================

assert WEIGHT_MODE in ("none", "inverse", "sqrt", "effective"), "bad WEIGHT_MODE"
assert TARGET_MODE in ("auto", "manual"), "bad TARGET_MODE"
assert BACKBONE in ("resnet50", "efficientnet_b3", "convnext_tiny", "swin_t", "transnet"), "bad BACKBONE"

PRESETS = {   # per-backbone defaults; a CONFIG value that is not None wins over these
    "resnet50":        dict(IMG_SIZE=224, LR=2e-4, WEIGHT_DECAY=1e-4, BATCH_SIZE=32, EPOCHS=25, GRAD_CLIP=2.0),
    "efficientnet_b3": dict(IMG_SIZE=300, LR=2e-4, WEIGHT_DECAY=1e-4, BATCH_SIZE=32, EPOCHS=25, GRAD_CLIP=2.0),
    "convnext_tiny":   dict(IMG_SIZE=224, LR=1e-4, WEIGHT_DECAY=0.05, BATCH_SIZE=32, EPOCHS=25, GRAD_CLIP=2.0),
    "swin_t":          dict(IMG_SIZE=224, LR=1e-4, WEIGHT_DECAY=0.05, BATCH_SIZE=32, EPOCHS=25, GRAD_CLIP=2.0),
    "transnet":        dict(IMG_SIZE=224, LR=1e-4, WEIGHT_DECAY=1e-4, BATCH_SIZE=32, EPOCHS=25, GRAD_CLIP=1.0),   # LR = ResNet-18 part
}
PRESET = PRESETS[BACKBONE]
for _k, _v in PRESET.items():
    if globals()[_k] is None:
        globals()[_k] = _v
assert BACKBONE != "transnet" or IMG_SIZE == 224, "transnet needs IMG_SIZE = 224 (7x7 = 49 tokens)"
assert TTA_MODE in ("flip", "d4") and MIX_PROB >= 0
assert AUG_LEVEL in ("base", "strong") and SELECT in ("best", "last")
REAL = dict(epochs=EPOCHS, mode=TARGET_MODE, cap=CAP_N, floor=FLOOR_N, max_repeat=MAX_REPEAT)   # kept for the dry-run time projection
if DRY_RUN:
    EPOCHS = 2
    TARGET_MODE, CAP_N, FLOOR_N, MAX_REPEAT = "auto", 100, 50, 6     # scaled down to match the tiny dry-run subset
    print("*** DRY RUN: tiny subset, 2 epochs, scaled-down targets. Numbers are meaningless. Set DRY_RUN = False for the real run. ***")
RUN_LABEL = ("rs" if RESAMPLE else "norm") + "_" + WEIGHT_MODE
print(f"Member: {MEMBER} | Run: {RUN_LABEL} | {BACKBONE} @ {IMG_SIZE}px | seed {SEED} | epochs {EPOCHS} | batch {BATCH_SIZE} | lr {LR}")
print(f"schedule {USE_SCHEDULE} | EMA {USE_EMA} | label smoothing {LABEL_SMOOTH} | aug {AUG_LEVEL} | preload {PRELOAD} | "
      f"select {SELECT} | TTA {USE_TTA} | post-hoc {POSTHOC}")
if RESAMPLE:
    print(f"Resampling: {TARGET_MODE} | cap {CAP_N} | floor {FLOOR_N} | max repeat {MAX_REPEAT} | "
          f"every epoch: {RESAMPLE_EACH_EPOCH} | lesion-aware: {LESION_AWARE} | dedup: {DEDUP} (T_DUP={T_DUP})")

## 1 · Splits and leakage check
The check below compares **lesion ids**, not just image ids. If it fails, the notebook stops on purpose.

In [ ]:
def check_no_leakage(train_df, val_df, test_df):
    pairs = [("train", "val", train_df, val_df), ("train", "test", train_df, test_df), ("val", "test", val_df, test_df)]
    rows = []
    for n1, n2, a, b in pairs:
        rows.append({"pair": f"{n1}-{n2}",
                     "shared_images": len(set(a["image_id"]) & set(b["image_id"])),
                     "shared_lesions": len(set(a["lesion_id"]) & set(b["lesion_id"]))})
    rep = pd.DataFrame(rows)
    print(rep.to_string(index=False))
    if (rep["shared_images"] > 0).any() or (rep["shared_lesions"] > 0).any():
        raise RuntimeError(
            "LEAKAGE: some lesions/images appear in more than one split, so test scores would be inflated. "
            "Do NOT trust results from these splits. Run the optional 'regroup' cell below to create "
            "lesion-grouped splits, point SPLIT_DIR at the new folder, and rerun (tell your team: the shared splits change).")
    print("OK: no image or lesion appears in more than one split.")
    return rep


def take_per_class(df, k, seed=0):
    # up to k rows per class (used only for the DRY_RUN smoke test); k may be an int or {class: int}
    d = df.sample(frac=1, random_state=seed)
    if isinstance(k, dict):
        return pd.concat([g.head(k.get(c, 0)) for c, g in d.groupby("dx")]).reset_index(drop=True)
    return d.groupby("dx").head(k).reset_index(drop=True)


def compute_class_weights(train_counts, mode, beta=0.999):
    n = np.asarray(train_counts, dtype=float)
    assert (n > 0).all(), "a class has zero training images"
    K, N = len(n), n.sum()
    if mode == "none":
        w = np.ones(K)
    elif mode == "inverse":              # w_c = N / (K * n_c)
        w = N / (K * n)
    elif mode == "sqrt":                 # w_c ~ 1/sqrt(n_c), rescaled to mean 1
        w = 1.0 / np.sqrt(n)
        w = w / w.mean()
    elif mode == "effective":            # Cui et al. 2019, normalised to sum K
        w = (1.0 - beta) / (1.0 - np.power(beta, n))
        w = w / w.sum() * K
    else:
        raise ValueError(mode)
    return w


def make_grouped_splits(metadata, out_dir, seed=42, n_splits=10):
    # ~80/10/10 split, stratified by class, grouped by lesion_id (a lesion never crosses splits)
    from sklearn.model_selection import StratifiedGroupKFold
    meta = metadata.reset_index(drop=True)
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold = np.full(len(meta), -1)
    for f, (_, idx) in enumerate(sgkf.split(meta, meta["dx"], groups=meta["lesion_id"])):
        fold[idx] = f
    test, val, train = meta[fold == 0], meta[fold == 1], meta[fold >= 2]
    check_no_leakage(train, val, test)
    os.makedirs(out_dir, exist_ok=True)
    train.to_csv(f"{out_dir}/train_split.csv", index=False)
    val.to_csv(f"{out_dir}/val_split.csv", index=False)
    test.to_csv(f"{out_dir}/test_split.csv", index=False)
    print("saved to", out_dir, "| sizes:", len(train), len(val), len(test))

In [ ]:
metadata = pd.read_csv(f"{DATA_DIR}/HAM10000_metadata.csv")
train_df = pd.read_csv(f"{SPLIT_DIR}/train_split.csv")
val_df   = pd.read_csv(f"{SPLIT_DIR}/val_split.csv")
test_df  = pd.read_csv(f"{SPLIT_DIR}/test_split.csv")
print("Metadata:", metadata.shape, "| Train:", train_df.shape, "| Val:", val_df.shape, "| Test:", test_df.shape)

check_no_leakage(train_df, val_df, test_df)      # stops the notebook if there is leakage

FULL_TRAIN_N, FULL_VAL_N, FULL_TEST_N = train_df["dx"].value_counts().to_dict(), len(val_df), len(test_df)   # for the dry-run projection

if DRY_RUN:                                      # tiny stratified subset, only for the smoke test
    # unequal on purpose, so the dry run exercises BOTH undersampling (nv) and oversampling (df, vasc, akiec, bcc)
    train_df = take_per_class(train_df, dict(nv=150, mel=60, bkl=60, bcc=40, akiec=30, vasc=20, df=20), SEED)
    val_df   = take_per_class(val_df, 16, SEED)
    test_df  = take_per_class(test_df, 16, SEED)
    print("DRY RUN subset sizes:", len(train_df), len(val_df), len(test_df))

### Optional: only run this if the leakage check above failed
It builds new lesion-grouped 80/10/10 splits and saves them to your Drive. Afterwards set `SPLIT_DIR` in the CONFIG cell to the printed folder and rerun from the top. Your teammates must use the same new files, and the first baseline notebook's numbers are then invalid.

In [ ]:
RUN_REGROUP = False     # set True ONLY if the leakage check failed
if RUN_REGROUP:
    make_grouped_splits(metadata, "/content/drive/MyDrive/HAM10000_model/splits_grouped", seed=SEED)

## 2 · Images, near-duplicates, per-epoch targets, class weights

In [ ]:
from glob import glob as _glob

IMAGE_DIRS = [os.path.join(DATA_DIR, "HAM10000_images_part_1"), os.path.join(DATA_DIR, "HAM10000_images_part_2")]

if COPY_LOCAL and not DRY_RUN:
    LOCAL_IMG = "/content/ham_images"
    if not os.path.isdir(LOCAL_IMG) or len(os.listdir(LOCAL_IMG)) < 10015:
        os.makedirs(LOCAL_IMG, exist_ok=True)
        for d in IMAGE_DIRS:
            !cp -r "{d}/." {LOCAL_IMG}/
    IMAGE_DIRS = [LOCAL_IMG]

image_paths = {}
for image_dir in IMAGE_DIRS:
    for path in _glob(os.path.join(image_dir, "*.jpg")):
        image_paths[os.path.splitext(os.path.basename(path))[0]] = path
print("Total JPG images found:", len(image_paths))

classes = sorted(metadata["dx"].unique())                       # same alphabetical order as the first baseline
class_to_idx = {c: i for i, c in enumerate(classes)}
print("Classes:", class_to_idx)

for df in (train_df, val_df, test_df):
    df["image_path"] = df["image_id"].map(image_paths)
    df["label"] = df["dx"].map(class_to_idx)
for name, df in (("train", train_df), ("val", val_df), ("test", test_df)):
    assert df["image_path"].notna().all() and df["label"].notna().all(), f"{name}: missing image or label"
print("All images and labels found.")

### 2a · Near-duplicate check
HAM10000 has several photos per lesion and a few near-identical image pairs. This step hashes every image (dHash, 256 bit) and compares all pairs.
- **distance ≤ `T_DUP`:** same picture. A train image that is a near-copy of a val/test image is **dropped from train** (otherwise test scores are inflated). Of two near-identical train images, one is dropped.
- **`T_DUP` < distance ≤ `T_REVIEW`:** grey zone. Shown below as pictures for **you to judge**; nothing is removed automatically.
- Val and test sets are never changed. If val/test contain near-copies of each other, it is reported only.
- Hashes are cached on Drive, so later runs are fast.

In [ ]:
from PIL import Image

# ======================= near-duplicate detection (dHash) =======================
def dhash_bits(path, size=16):
    """256-bit difference hash: grayscale, resize to (size+1) x size, compare horizontal neighbours."""
    im = Image.open(path)
    im.draft("L", (size * 8, size * 8))          # cheaper JPEG decode; hash stays consistent across runs
    a = np.asarray(im.convert("L").resize((size + 1, size), Image.BILINEAR), dtype=np.int16)
    return (a[:, 1:] > a[:, :-1]).flatten()


def compute_hashes(df, cache_path):
    cache = {}
    if os.path.exists(cache_path):
        z = np.load(cache_path)
        cache = dict(zip(z["ids"].tolist(), z["bits"]))
    todo = [(i, p) for i, p in zip(df["image_id"], df["image_path"]) if i not in cache]
    for i, p in todo:
        cache[i] = dhash_bits(p)
    if todo:
        os.makedirs(os.path.dirname(cache_path), exist_ok=True)
        np.savez(cache_path, ids=np.array(list(cache.keys())), bits=np.stack(list(cache.values())))
    return np.stack([cache[i] for i in df["image_id"]])


def near_pairs(bits, max_dist, chunk=1000):
    """All index pairs (i<j) with Hamming distance <= max_dist."""
    X = bits.astype(np.float32) * 2 - 1
    n, d = X.shape
    out = []
    for s in range(0, n, chunk):
        dist = (d - X[s:s + chunk] @ X.T) / 2
        ii, jj = np.nonzero(dist <= max_dist)
        for a, b in zip(ii + s, jj):
            if a < b:
                out.append((int(a), int(b), int(round(dist[a - s, b]))))
    return pd.DataFrame(out, columns=["i", "j", "dist"]).astype(int)


def plan_dedup(all_df, pairs, t_dup):
    """Returns (annotated pairs, set of TRAIN image_ids to drop). Val/test are never dropped."""
    sp, ids, dx = all_df["split"].to_numpy(), all_df["image_id"].to_numpy(), all_df["dx"].to_numpy()
    p = pairs.copy()
    p["split_i"], p["split_j"] = sp[p["i"].to_numpy()], sp[p["j"].to_numpy()]
    p["id_i"], p["id_j"] = ids[p["i"].to_numpy()], ids[p["j"].to_numpy()]
    p["dx_i"], p["dx_j"] = dx[p["i"].to_numpy()], dx[p["j"].to_numpy()]
    p["same_dx"] = p["dx_i"] == p["dx_j"]
    p["zone"] = np.where(p["dist"] <= t_dup, "duplicate", "review")
    drop = set()
    dup = p[p["zone"] == "duplicate"]
    for r in dup.itertuples():                   # train image that is a near-copy of a val/test image
        if r.split_i == "train" and r.split_j != "train":
            drop.add(r.id_i)
        elif r.split_j == "train" and r.split_i != "train":
            drop.add(r.id_j)
    for r in dup[(dup["split_i"] == "train") & (dup["split_j"] == "train")].sort_values("dist").itertuples():
        if r.id_i not in drop and r.id_j not in drop:   # train-train: keep one, drop the other
            drop.add(r.id_j)
    return p, drop


# ======================= per-epoch targets and resampling =======================
def compute_targets(counts, mode, cap, floor, max_repeat, manual=None, classes=None):
    """Images per class per epoch. auto: big -> cap, small -> up to floor (but at most max_repeat x), middle untouched."""
    n = np.asarray(counts, dtype=int)
    if mode == "manual":
        t = np.array([manual[c] for c in classes], dtype=int)
    else:
        assert floor <= cap, "FLOOR_N must be <= CAP_N"
        t = n.copy()
        t[n > cap] = cap
        small = n < floor
        t[small] = np.minimum(floor, n[small] * max_repeat)
    return np.maximum(t, 1)


def build_epoch_indices(labels, lesion_ids, targets, rng, lesion_aware=True):
    """Row positions to use this epoch. Undersample: draw without replacement (optionally favouring lesions with
    few photos). Oversample: every image k times, plus a random remainder, so repeats are as even as possible."""
    labels, lesion_ids = np.asarray(labels), np.asarray(lesion_ids)
    parts = []
    for c, t in enumerate(targets):
        idx = np.flatnonzero(labels == c)
        n = len(idx)
        if t <= n:
            if lesion_aware:
                _, inv, cnt = np.unique(lesion_ids[idx], return_inverse=True, return_counts=True)
                p = 1.0 / cnt[inv]
                pick = rng.choice(idx, size=int(t), replace=False, p=p / p.sum())
            else:
                pick = rng.choice(idx, size=int(t), replace=False)
        else:
            k, r = divmod(int(t), n)
            pick = np.tile(idx, k)
            if r:
                pick = np.concatenate([pick, rng.choice(idx, size=r, replace=False)])
        parts.append(pick)
    out = np.concatenate(parts)
    rng.shuffle(out)
    return out


def describe_epoch(idx, labels, lesion_ids, classes, counts, targets, weights):
    labels, lesion_ids, rows = np.asarray(labels), np.asarray(lesion_ids), []
    for c, name in enumerate(classes):
        sel = idx[labels[idx] == c]
        u = len(np.unique(sel))
        rows.append(dict(cls=name, train_n=int(counts[c]), per_epoch=len(sel), pct_of_epoch=round(100 * len(sel) / len(idx), 1),
                         unique_imgs=u, unique_lesions=len(np.unique(lesion_ids[sel])),
                         avg_repeat=round(len(sel) / max(1, u), 2), weight=round(float(weights[c]), 3)))
    return pd.DataFrame(rows)

In [ ]:
import matplotlib.pyplot as plt

def show_pairs(p, all_df, n, title):
    if len(p) == 0:
        print(f"{title}: none"); return
    p = p.head(n)
    fig, ax = plt.subplots(2, len(p), figsize=(2.6 * len(p), 5.6), squeeze=False)
    for k, r in enumerate(p.itertuples()):
        for row, (col, idc, dxc, spc) in enumerate((("i", "id_i", "dx_i", "split_i"), ("j", "id_j", "dx_j", "split_j"))):
            ax[row, k].imshow(Image.open(all_df.loc[getattr(r, col), "image_path"]).convert("RGB"))
            ax[row, k].set_title(f"{getattr(r, dxc)} · {getattr(r, spc)}" + (f"\nd={r.dist}" if row == 0 else ""), fontsize=8)
            ax[row, k].axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()


DEDUP_STATS = {"enabled": bool(DEDUP)}
if DEDUP:
    t0 = time.time()
    all_df = pd.concat([train_df.assign(split="train"), val_df.assign(split="val"), test_df.assign(split="test")], ignore_index=True)
    bits = compute_hashes(all_df, f"{OUT_DIR}/dhash_cache.npz")
    print(f"hashed {len(all_df)} images in {time.time() - t0:.0f}s")
    pairs, drop_ids = plan_dedup(all_df, near_pairs(bits, T_REVIEW), T_DUP)
    os.makedirs(OUT_DIR, exist_ok=True)
    pairs.drop(columns=["i", "j"]).to_csv(f"{OUT_DIR}/near_duplicate_pairs{'_DRYRUN' if DRY_RUN else ''}.csv", index=False)

    summ = (pairs.assign(combo=lambda d: d["split_i"] + "-" + d["split_j"])
                 .groupby(["zone", "combo"]).size().rename("pairs").reset_index())
    print(summ.to_string(index=False) if len(summ) else "No near-duplicate pairs found.")
    dup = pairs[pairs["zone"] == "duplicate"]
    n_diff = int((~dup["same_dx"]).sum())
    vt = int(((dup["split_i"] != "train") & (dup["split_j"] != "train")).sum())
    print(f"duplicate pairs with DIFFERENT labels (label noise, look at them): {n_diff}")
    print(f"duplicate pairs between val/test (reported only, not changed): {vt}")

    before = len(train_df)
    train_df = train_df[~train_df["image_id"].isin(drop_ids)].reset_index(drop=True)
    print(f"train images removed as near-duplicates: {before - len(train_df)}  ({before} -> {len(train_df)})")
    DEDUP_STATS.update(t_dup=T_DUP, t_review=T_REVIEW, duplicate_pairs=int(len(dup)), diff_label_pairs=n_diff,
                       val_test_duplicate_pairs=vt, train_removed=int(before - len(train_df)))
    show_pairs(dup.sort_values("dist", ascending=False), all_df, 6, "Auto-removed duplicates (most different first): do these look like the same picture?")
    show_pairs(pairs[pairs["zone"] == "review"].sort_values("dist"), all_df, 6, "Grey zone (NOT removed): same lesion/picture or just similar?")
else:
    print("DEDUP = False: skipping.")

In [ ]:
counts = train_df["label"].value_counts().reindex(range(len(classes))).fillna(0).astype(int).values
assert (counts > 0).all(), "a class has no training images left"

if RESAMPLE:
    targets = compute_targets(counts, TARGET_MODE, CAP_N, FLOOR_N, MAX_REPEAT, MANUAL_TARGETS, classes)
else:
    targets = counts.copy()
class_weights = compute_class_weights(targets, WEIGHT_MODE, EFF_BETA)     # from what the model sees per epoch

# simulate ONE epoch to show exactly what resampling does
if RESAMPLE:
    _idx = build_epoch_indices(train_df["label"].to_numpy(), train_df["lesion_id"].to_numpy(), targets,
                               np.random.default_rng(SEED), LESION_AWARE)
else:
    _idx = np.arange(len(train_df))
table = describe_epoch(_idx, train_df["label"].to_numpy(), train_df["lesion_id"].to_numpy(), classes, counts, targets, class_weights)
print(f"RESAMPLE = {RESAMPLE} | WEIGHT_MODE = {WEIGHT_MODE}  (weights come from per-epoch counts, TRAIN only)")
print(table.to_string(index=False))
print(f"images per epoch: {len(_idx)} (raw train set: {len(train_df)})")
print(f"imbalance ratio (largest/smallest class): raw {counts.max() / counts.min():.1f} -> per epoch {targets.max() / targets.min():.1f}")
print(f"largest/smallest weight ratio: {class_weights.max() / class_weights.min():.1f}")
if RESAMPLE and (targets / counts).max() > 8:
    print("WARNING: some class is repeated more than 8x on average; expect overfitting to those few images.")

## 3 · Data, model, loss

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights, efficientnet_b3, EfficientNet_B3_Weights, convnext_tiny, ConvNeXt_Tiny_Weights
from torchvision.models import swin_t, Swin_T_Weights, resnet18, ResNet18_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class TransNet(nn.Module):
    """ResNet-18 (no avgpool/fc) -> 49 tokens -> Linear 512->96 + learned position embedding -> 6 pre-norm transformer layers -> mean-pool -> Linear 96->k."""
    def __init__(self, k, pretrained=True):
        super().__init__()
        r = resnet18(weights=ResNet18_Weights.DEFAULT if pretrained else None)
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu, r.maxpool, r.layer1, r.layer2, r.layer3, r.layer4)
        self.proj = nn.Linear(512, 96)
        self.pos = nn.Parameter(torch.zeros(1, 49, 96)); nn.init.trunc_normal_(self.pos, std=0.02)
        layer = nn.TransformerEncoderLayer(d_model=96, nhead=8, dim_feedforward=384, dropout=0.1, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=6, enable_nested_tensor=False)
        self.head = nn.Linear(96, k)
    def forward(self, x, verbose=False):
        f = self.stem(x)
        if verbose: print("  after ResNet-18        :", tuple(f.shape))
        t = f.flatten(2).transpose(1, 2)
        if verbose: print("  after flatten to tokens:", tuple(t.shape))
        t = self.proj(t) + self.pos
        if verbose: print("  after Linear + pos-emb :", tuple(t.shape))
        t = self.encoder(t).mean(1)
        out = self.head(t)
        if verbose: print("  after encoder+pool+head:", tuple(out.shape))
        return out


def build_model(name, k, pretrained=True):
    """ImageNet-pretrained CNN with a fresh k-class head (the only layer replaced)."""
    if name == "resnet50":
        m = resnet50(weights=ResNet50_Weights.DEFAULT if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, k)
    elif name == "efficientnet_b3":
        m = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT if pretrained else None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, k)       # keeps torchvision's dropout 0.3 before it
    elif name == "convnext_tiny":
        m = convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None)
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, k)       # keeps LayerNorm2d + Flatten before it
    elif name == "swin_t":
        m = swin_t(weights=Swin_T_Weights.IMAGENET1K_V1 if pretrained else None)
        m.head = nn.Linear(m.head.in_features, k)                          # Linear(768, k)
    elif name == "transnet":
        m = TransNet(k, pretrained)
    else:
        raise ValueError(name)
    return m


def _probe_step(bs):
    """Two real training steps (model + EMA copy + AdamW, mixed precision) at the true image size. Returns peak GPU GB."""
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    m = build_model(BACKBONE, len(classes), pretrained=False).to(device).train()
    shadow = copy.deepcopy(m) if USE_EMA else None
    opt = torch.optim.AdamW(m.parameters(), lr=1e-5)
    sc = torch.amp.GradScaler("cuda", enabled=True)
    x = torch.randn(bs, 3, IMG_SIZE, IMG_SIZE, device=device); y = torch.randint(0, len(classes), (bs,), device=device)
    for _ in range(2):
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            loss = nn.functional.cross_entropy(m(x), y)
        sc.scale(loss).backward(); sc.step(opt); sc.update()
    torch.cuda.synchronize()
    peak = torch.cuda.max_memory_allocated() / 1e9
    del m, shadow, opt, sc, x, y, loss
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    return peak


def pick_batch_size(preferred, fallback=24, step=None):
    """Use the preferred batch size unless it runs out of GPU memory; then try the fallback. Returns (batch_size, probe_peak_GB)."""
    if step is None:
        if device.type != "cuda":
            return preferred, None
        step = _probe_step
    for bs in dict.fromkeys([preferred, fallback] if fallback < preferred else [preferred]):
        try:
            return bs, step(bs)
        except torch.cuda.OutOfMemoryError:
            print(f"  CUDA out of memory at batch size {bs}")
            if device.type == "cuda":
                torch.cuda.empty_cache()
    raise RuntimeError("Even the fallback batch size ran out of GPU memory. Lower IMG_SIZE or restart the runtime.")


BATCH_SIZE_REQUESTED = BATCH_SIZE
BATCH_SIZE, PROBE_PEAK_GB = pick_batch_size(BATCH_SIZE)
print(f"Batch size: {BATCH_SIZE}" + (" (fallback: the preferred size ran out of memory)" if BATCH_SIZE != BATCH_SIZE_REQUESTED else "")
      + (f" | probe peak GPU memory {PROBE_PEAK_GB:.1f} GB (model + EMA + optimizer + one batch)" if PROBE_PEAK_GB else ""))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
print("Device:", device, "| mixed precision:", USE_AMP)

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
if AUG_LEVEL == "strong":
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.65, 1.0), ratio=(0.85, 1.18)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.03),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.10), value=0),
    ])
else:                                                  # same as v3
    train_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


def shades_of_gray(a, p=6):
    """Illumination normalisation (Minkowski p-norm). A neutral image stays unchanged."""
    f = a.astype(np.float32)
    illum = (f ** p).mean((0, 1)) ** (1.0 / p)
    illum = illum / (np.linalg.norm(illum) + 1e-6)
    return np.clip(f / (illum * np.sqrt(3.0) + 1e-6), 0, 255).astype(np.uint8)


def preload_images(frames, size, workers=8):
    """Decode + resize every needed image once; training then skips JPEG decoding."""
    paths = {}
    for d in frames:
        paths.update(zip(d["image_id"], d["image_path"]))
    ids = list(paths)
    arr = np.empty((len(ids), size, size, 3), dtype=np.uint8)
    def load(k):
        im = Image.open(paths[ids[k]])
        im.draft("RGB", (size, size))
        a_ = np.asarray(im.convert("RGB").resize((size, size), Image.BILINEAR))
        arr[k] = shades_of_gray(a_) if COLOR_CONSTANCY else a_
    with ThreadPoolExecutor(workers) as ex:
        list(ex.map(load, range(len(ids))))
    return {"array": arr, "index": {i: k for k, i in enumerate(ids)}}


def free_ram_gb():
    try:
        import psutil
        return psutil.virtual_memory().available / 1e9
    except Exception:
        try:
            for line in open("/proc/meminfo"):
                if line.startswith("MemAvailable"):
                    return int(line.split()[1]) / 1e6
        except Exception:
            pass
    return None


STORE, PRELOAD_S, PRELOAD_N = None, 0.0, 0
if PRELOAD:
    n_unique = len(set(train_df["image_id"]) | set(val_df["image_id"]) | set(test_df["image_id"]))
    need_gb, free_gb = n_unique * IMG_SIZE * IMG_SIZE * 3 / 1e9, free_ram_gb()
    if free_gb is None or free_gb < 5 or free_gb < need_gb + 2:
        PRELOAD = False
        print(f"PRELOAD switched OFF: free RAM {'unknown' if free_gb is None else f'{free_gb:.1f} GB'}, "
              f"needs >= 5 GB and >= {need_gb + 2:.1f} GB (images {need_gb:.1f} GB + 2 GB spare). Images will be read from disk each epoch (slower).")
    else:
        print(f"RAM guard ok: {free_gb:.1f} GB free, preload needs {need_gb:.1f} GB")
assert PRELOAD or not COLOR_CONSTANCY, "COLOR_CONSTANCY needs PRELOAD = True (and enough free RAM)"
if PRELOAD:
    t0 = time.time()
    STORE = preload_images([train_df, val_df, test_df], IMG_SIZE)
    PRELOAD_S, PRELOAD_N = time.time() - t0, len(STORE["index"])
    print(f"Preloaded {len(STORE['index'])} images at {IMG_SIZE}px into RAM ({STORE['array'].nbytes / 1e9:.1f} GB) in {time.time() - t0:.0f}s")


class HAM10000Dataset(Dataset):
    def __init__(self, dataframe, transform=None, store=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform, self.store = transform, store
        if store is not None:
            self.rows = np.array([store["index"][i] for i in self.df["image_id"]])
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        if self.store is not None:
            image = Image.fromarray(self.store["array"][self.rows[idx]])
        else:
            image = Image.open(self.df.iloc[idx]["image_path"]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, int(self.df.iloc[idx]["label"])

def seed_worker(worker_id):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)

g = torch.Generator(); g.manual_seed(SEED)
kw = dict(batch_size=BATCH_SIZE, num_workers=2, pin_memory=(device.type == "cuda"), worker_init_fn=seed_worker)

class EpochResampler(torch.utils.data.Sampler):
    """Gives DataLoader a fresh class-rebalanced list of row positions every epoch (see build_epoch_indices)."""
    def __init__(self, labels, lesion_ids, targets, seed, lesion_aware=True, every_epoch=True):
        self.labels, self.lesion_ids, self.targets = np.asarray(labels), np.asarray(lesion_ids), np.asarray(targets)
        self.seed, self.lesion_aware, self.every_epoch, self.epoch = seed, lesion_aware, every_epoch, 0
    def __len__(self):
        return int(self.targets.sum())
    def __iter__(self):
        ep = self.epoch if self.every_epoch else 0
        self.epoch += 1
        rng = np.random.default_rng(self.seed * 1000 + ep)
        return iter(build_epoch_indices(self.labels, self.lesion_ids, self.targets, rng, self.lesion_aware).tolist())

train_ds = HAM10000Dataset(train_df, train_tf, STORE)  # train_df was reset_index'd, so row position == dataset index
if RESAMPLE:
    sampler = EpochResampler(train_df["label"].to_numpy(), train_df["lesion_id"].to_numpy(), targets, SEED,
                             LESION_AWARE, RESAMPLE_EACH_EPOCH)
    train_loader = DataLoader(train_ds, sampler=sampler, **kw)
else:
    train_loader = DataLoader(train_ds, shuffle=True, generator=g, **kw)
val_loader  = DataLoader(HAM10000Dataset(val_df, eval_tf, STORE), shuffle=False, **kw)
test_loader = DataLoader(HAM10000Dataset(test_df, eval_tf, STORE), shuffle=False, **kw)
print("Training images seen per epoch:", len(sampler) if RESAMPLE else len(train_ds))
print("Train / val / test batches:", len(train_loader), len(val_loader), len(test_loader))

In [ ]:
# ---- run tag (decided now, because the checkpoint file is named after it) ----
feats = "-".join(n for n, on in (("sch", USE_SCHEDULE), ("ema", USE_EMA), ("tta", USE_TTA), ("ph", POSTHOC)) if on)
_dev = []                       # every setting that differs from its default is written into the tag
if LR != PRESET["LR"]: _dev.append(f"lr{LR:g}")
if WEIGHT_DECAY != PRESET["WEIGHT_DECAY"]: _dev.append(f"wd{WEIGHT_DECAY:g}")
if BATCH_SIZE != PRESET["BATCH_SIZE"]: _dev.append(f"bs{BATCH_SIZE}")
if WARMUP_EPOCHS != 1: _dev.append(f"wu{WARMUP_EPOCHS}")
if EMA_DECAY != 0.998: _dev.append(f"emad{EMA_DECAY:g}")
if GRAD_CLIP != 2.0: _dev.append(f"clip{GRAD_CLIP:g}")
if RESAMPLE and MAX_REPEAT != 6: _dev.append(f"rep{MAX_REPEAT}")
if RESAMPLE and not LESION_AWARE: _dev.append("nolesion")
if RESAMPLE and not RESAMPLE_EACH_EPOCH: _dev.append("fixedsample")
if not DEDUP: _dev.append("nodedup")
elif T_DUP != 12: _dev.append(f"tdup{T_DUP}")
if MIX_PROB > 0 and (MIXUP_ALPHA > 0 or CUTMIX_ALPHA > 0): _dev.append(f"mix{MIX_PROB:g}m{MIXUP_ALPHA:g}c{CUTMIX_ALPHA:g}")
if COLOR_CONSTANCY: _dev.append("cc")
if USE_TTA and TTA_MODE == "d4": _dev.append("d4")
tag = (f"v6_{BACKBONE}{IMG_SIZE}_{RUN_LABEL}"
       + (f"_{TARGET_MODE}" + (f"_cap{CAP_N}_floor{FLOOR_N}" if TARGET_MODE == "auto" else "") if RESAMPLE else "")
       + f"_{AUG_LEVEL}_ls{LABEL_SMOOTH}_{feats or 'plain'}_ep{EPOCHS}_{SELECT}"
       + ("".join("_" + d for d in _dev)) + ("_DRYRUN" if DRY_RUN else "") + f"_seed{SEED}")
os.makedirs(OUT_DIR, exist_ok=True)
CKPT = f"{OUT_DIR}/ckpt_{tag}.pt"
print("Run tag:", tag)


class EMA:
    """Exponential moving average of ALL weights and batch-norm statistics. The averaged copy is what we evaluate."""
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = copy.deepcopy(model).eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)
        self.e, self.m = self.shadow.state_dict(), model.state_dict()     # tensors share storage with the live ones
    @torch.no_grad()
    def update(self, step):
        d = min(self.decay, (1 + step) / (10 + step))                    # warm-up so early averages are not stuck at init
        for k, v in self.e.items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(self.m[k].detach(), alpha=1 - d)
            else:
                v.copy_(self.m[k])


model = build_model(BACKBONE, len(classes)).to(device)
PARAMS_M = sum(p.numel() for p in model.parameters()) / 1e6
print(f"parameters: {PARAMS_M:.2f} M")
if DRY_RUN and BACKBONE == "transnet":
    print("TransNet tensor shapes for one batch:")
    with torch.no_grad():
        model.eval(); model(torch.zeros(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE, device=device), verbose=True); model.train()
ema = EMA(model, EMA_DECAY) if USE_EMA else None
eval_net = ema.shadow if USE_EMA else model

weight_tensor = torch.tensor(class_weights, dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=weight_tensor, label_smoothing=LABEL_SMOOTH)    # mode "none" -> all-ones weights
if BACKBONE in ("convnext_tiny", "swin_t"):
    # weight decay 0.05 is meant for weight matrices/kernels only, not for norm scales, biases or layer-scale vectors
    decay, no_decay = [], []
    for n_, p_ in model.named_parameters():
        (no_decay if p_.ndim <= 1 or n_.endswith(".bias") or "layer_scale" in n_ or "relative_position_bias" in n_ else decay).append(p_)
    param_groups = [dict(params=decay, weight_decay=WEIGHT_DECAY), dict(params=no_decay, weight_decay=0.0)]
elif BACKBONE == "transnet":
    # ResNet-18 part at LR, new layers at LR_NEW; weight decay on weight matrices/kernels only
    g_ = {(a, b): [] for a in ("cnn", "new") for b in ("decay", "nodecay")}
    for n_, p_ in model.named_parameters():
        g_[("cnn" if n_.startswith("stem.") else "new", "nodecay" if p_.ndim <= 1 or n_.endswith(".bias") or n_ == "pos" else "decay")].append(p_)
    param_groups = [dict(params=ps, lr=(LR if part == "cnn" else LR_NEW), weight_decay=(WEIGHT_DECAY if kind == "decay" else 0.0))
                    for (part, kind), ps in g_.items() if ps]
else:
    param_groups = [dict(params=list(model.parameters()), weight_decay=WEIGHT_DECAY)]
optimizer = torch.optim.AdamW(param_groups, lr=LR)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

steps_per_epoch = len(train_loader)
total_steps, warm_steps = EPOCHS * steps_per_epoch, WARMUP_EPOCHS * steps_per_epoch
def lr_lambda(step):
    if warm_steps and step < warm_steps:
        return (step + 1) / warm_steps
    prog = (step - warm_steps) / max(1, total_steps - warm_steps)
    return 0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))      # decays to 2% of the peak rate
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda) if USE_SCHEDULE else None

print(f"Model: {BACKBONE} (ImageNet pretrained) | loss: weighted CE ({WEIGHT_MODE}), label smoothing {LABEL_SMOOTH}")
print(f"Optimizer: AdamW | peak lr {LR} | weight decay {WEIGHT_DECAY} | schedule {USE_SCHEDULE} | EMA {USE_EMA} | steps/epoch {steps_per_epoch}")

## 4 · Training
Each epoch is validated with the (EMA-smoothed, if on) weights. The kept checkpoint is the epoch with the highest **validation macro-F1** (`SELECT = "best"`) or simply the last one (`SELECT = "last"`). The validation loss shown is *unweighted* cross-entropy. The training loss is the loss being optimised (weighted, label-smoothed), so it should not be compared across settings.

In [ ]:
from sklearn.metrics import f1_score, balanced_accuracy_score, recall_score, roc_auc_score

K = len(classes)
mel_idx = class_to_idx["mel"]
_t = lambda x: x.transpose(2, 3)
FLIPS = [lambda x: x, lambda x: x.flip(3), lambda x: x.flip(2), lambda x: x.flip((2, 3))]
if TTA_MODE == "d4":      # the 8 symmetries of a square image
    FLIPS += [_t, lambda x: _t(x).flip(3), lambda x: _t(x).flip(2), lambda x: _t(x).flip((2, 3))]

@torch.no_grad()
def predict(loader, net, tta=False):
    """Logits for every image. tta=True averages the logits over the flips (4, or 8 with TTA_MODE = 'd4')."""
    net.eval()
    logits, ys = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        outs = []
        for f in (FLIPS if tta else FLIPS[:1]):
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                outs.append(net(f(x)).float())
        logits.append(torch.stack(outs).mean(0).cpu()); ys.append(y)
    return torch.cat(logits), torch.cat(ys)


def summarize(logits, y):
    """All the metrics we report, from numpy logits and labels."""
    pred = logits.argmax(1)
    probs = torch.softmax(torch.tensor(logits), 1).numpy()
    try:
        auc = roc_auc_score(y, probs, multi_class="ovr", average="macro", labels=list(range(K)))
    except ValueError:
        auc = float("nan")
    return dict(acc=float((pred == y).mean()), bal=balanced_accuracy_score(y, pred),
                mf1=f1_score(y, pred, average="macro", labels=list(range(K)), zero_division=0),
                recalls=recall_score(y, pred, labels=list(range(K)), average=None, zero_division=0), auc=auc, pred=pred)


def tune_bias(logits, y, mel_floor=None, grid=np.arange(-1.5, 1.51, 0.25), passes=3, min_gain=1e-3):
    """Per-class additive offsets on the logits that maximise macro-F1 (on VALIDATION data), found by coordinate ascent.
    Coarse grid, limited range and a minimum gain per change keep it from fitting validation noise too hard."""
    def score(b):
        pred = (logits + b).argmax(1)
        s = f1_score(y, pred, average="macro", labels=list(range(K)), zero_division=0)
        if mel_floor is not None:
            rec = recall_score(y, pred, labels=[mel_idx], average=None, zero_division=0)[0]
            if rec < mel_floor:
                s -= 10 * (mel_floor - rec)
        return s
    b = np.zeros(K)
    base = best = score(b)
    for _ in range(passes):
        improved = False
        for c in range(K):
            for gv in grid:
                if gv == b[c]:
                    continue
                nb = b.copy(); nb[c] = gv
                s = score(nb)
                if s > best + min_gain:
                    best, b, improved = s, nb, True
        if not improved:
            break
    return b, base, best


# ---------------------------------------------------------------- checkpoint (every epoch) and auto-resume
def save_ckpt(epoch):
    obj = dict(tag=tag, epoch=epoch, model=model.state_dict(), ema=(ema.shadow.state_dict() if ema is not None else None),
               opt=optimizer.state_dict(), sched=(scheduler.state_dict() if scheduler is not None else None),
               scaler=scaler.state_dict(), history=history, timing=timing, best_f1=best_f1, best_epoch=best_epoch,
               best_state=best_state, global_step=global_step, sampler_epoch=(sampler.epoch if RESAMPLE else None),
               rng=dict(py=random.getstate(), np=np.random.get_state(), torch=torch.get_rng_state(), gen=g.get_state(),
                        cuda=(torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)))
    torch.save(obj, CKPT + ".tmp")
    os.replace(CKPT + ".tmp", CKPT)             # atomic: a disconnect mid-write never leaves a broken checkpoint


history, best_f1, best_epoch, best_state, timing = [], -1.0, 0, None, []
global_step, start_epoch = 0, 1
if RESUME and os.path.exists(CKPT):
    try:
        ck = torch.load(CKPT, map_location="cpu", weights_only=False)
        assert ck["tag"] == tag
        model.load_state_dict(ck["model"])
        if ema is not None: ema.shadow.load_state_dict(ck["ema"])
        optimizer.load_state_dict(ck["opt"])
        if scheduler is not None: scheduler.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"])
        history, timing, best_f1, best_epoch, best_state = ck["history"], ck["timing"], ck["best_f1"], ck["best_epoch"], ck["best_state"]
        global_step, start_epoch = ck["global_step"], ck["epoch"] + 1
        if RESAMPLE: sampler.epoch = ck["sampler_epoch"]
        r = ck["rng"]
        random.setstate(r["py"]); np.random.set_state(r["np"]); torch.set_rng_state(r["torch"]); g.set_state(r["gen"])
        if r["cuda"] is not None: torch.cuda.set_rng_state_all(r["cuda"])
        del ck
        print(f"RESUMED from {CKPT}: {start_epoch - 1} epoch(s) already done, continuing at epoch {start_epoch}")
    except Exception as e:
        raise RuntimeError("Checkpoint exists but does not match / is unreadable. Delete it or set RESUME = False.") from e

if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

for epoch in range(start_epoch, EPOCHS + 1):
    t0 = time.time()
    model.train()
    loss_sum, n = 0.0, 0
    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            if MIX_PROB > 0 and np.random.rand() < MIX_PROB and (MIXUP_ALPHA > 0 or CUTMIX_ALPHA > 0):
                use_cut = CUTMIX_ALPHA > 0 and (MIXUP_ALPHA <= 0 or np.random.rand() < 0.5)
                lam = float(np.random.beta(CUTMIX_ALPHA, CUTMIX_ALPHA) if use_cut else np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA))
                perm = torch.randperm(images.size(0), device=images.device)
                if use_cut:
                    H_, W_ = images.shape[2], images.shape[3]
                    ch, cw = int(H_ * (1 - lam) ** 0.5), int(W_ * (1 - lam) ** 0.5)
                    cy, cx = np.random.randint(H_), np.random.randint(W_)
                    y0, y1, x0, x1 = max(cy - ch // 2, 0), min(cy + ch // 2, H_), max(cx - cw // 2, 0), min(cx + cw // 2, W_)
                    images = images.clone(); images[:, :, y0:y1, x0:x1] = images[perm][:, :, y0:y1, x0:x1]
                    lam = 1 - (y1 - y0) * (x1 - x0) / (H_ * W_)               # exact kept share
                else:
                    images = lam * images + (1 - lam) * images[perm]
                outputs = model(images)
                loss = lam * criterion(outputs, labels) + (1 - lam) * criterion(outputs, labels[perm])
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()
        if ema is not None:
            ema.update(global_step)
        global_step += 1
        loss_sum += loss.item() * labels.size(0)
        n += labels.size(0)
    train_loss = loss_sum / n
    t_train = time.time() - t0

    t1 = time.time()
    val_logits, val_y = predict(val_loader, eval_net)
    t_val = time.time() - t1
    val_loss = F.cross_entropy(val_logits, val_y).item()          # unweighted, comparable across runs
    pred, yv = val_logits.argmax(1).numpy(), val_y.numpy()
    val_f1 = f1_score(yv, pred, average="macro", labels=list(range(K)), zero_division=0)
    val_bal = balanced_accuracy_score(yv, pred)
    val_mel = recall_score(yv, pred, labels=[mel_idx], average=None, zero_division=0)[0]
    history.append(dict(epoch=epoch, lr=optimizer.param_groups[0]["lr"], train_loss=train_loss, val_loss=val_loss,
                        val_acc=float((pred == yv).mean()), val_bal_acc=val_bal, val_macro_f1=val_f1, val_mel_recall=val_mel))

    star = ""
    if SELECT == "last" or val_f1 > best_f1:
        best_f1, best_epoch = val_f1, epoch
        best_state = {k: v.detach().cpu().clone() for k, v in eval_net.state_dict().items()}
        star = "  <- kept" if SELECT == "best" else ""
    timing.append(dict(epoch=epoch, train_s=t_train, val_s=t_val, imgs=n, ckpt_s=0.0))
    t2 = time.time()
    save_ckpt(epoch)
    timing[-1]["ckpt_s"] = time.time() - t2
    print(f"Epoch {epoch:02d}/{EPOCHS} | lr {optimizer.param_groups[0]['lr']:.1e} | train loss {train_loss:.4f} | val loss {val_loss:.4f} | "
          f"val macro-F1 {val_f1:.4f} | val bal-acc {val_bal:.4f} | val mel recall {val_mel:.3f} | "
          f"{t_train:.0f}s train + {t_val:.0f}s val + {timing[-1]['ckpt_s']:.0f}s ckpt{star}")
    if not DRY_RUN and epoch in (2, 3):
        eta = np.mean([t["train_s"] + t["val_s"] + t["ckpt_s"] for t in timing[1:] or timing]) * EPOCHS / 60
        print(f"   projected training time for all {EPOCHS} epochs: about {eta:.0f} min" + ("  <-- over the 40 min budget; consider EPOCHS=20" if eta > 40 else ""))

run_min = sum(t["train_s"] + t["val_s"] + t.get("ckpt_s", 0) for t in timing) / 60
PEAK_GB = torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else None
print(f"\nKept epoch: {best_epoch} (val macro-F1 {best_f1:.4f}) | training time {run_min:.1f} min (summed over epochs)"
      + (f" | peak GPU memory {PEAK_GB:.1f} GB allocated, {torch.cuda.max_memory_reserved() / 1e9:.1f} GB reserved" if PEAK_GB else ""))
hist_df = pd.DataFrame(history)
SEC_PER_EPOCH = float(np.mean([t['train_s'] + t['val_s'] for t in timing]))
print(f"Parameters: {PARAMS_M:.2f} M | seconds per epoch (train + validation): {SEC_PER_EPOCH:.0f}")

# ---------------------------------------------------------------- dry run: timing + projection of the FULL run
PROJECTION = None
if DRY_RUN and len(timing) >= 2:
    tt = timing[-1]                                        # last epoch: the first one includes GPU/cuDNN warm-up
    real_counts = np.array([FULL_TRAIN_N[c] for c in classes])
    real_imgs = int(compute_targets(real_counts, REAL["mode"], REAL["cap"], REAL["floor"], REAL["max_repeat"], MANUAL_TARGETS, classes).sum()) if RESAMPLE else int(real_counts.sum())
    s_img, v_img = tt["train_s"] / tt["imgs"], tt["val_s"] / len(val_df)
    pre = PRELOAD_S / max(1, PRELOAD_N) * (int(real_counts.sum()) + FULL_VAL_N + FULL_TEST_N) if PRELOAD else 0.0
    per_epoch = s_img * real_imgs + v_img * FULL_VAL_N + tt["ckpt_s"]
    final_eval = v_img * (FULL_VAL_N + FULL_TEST_N) * (4 if USE_TTA else 1)
    total_min = (pre + per_epoch * REAL["epochs"] + final_eval) / 60
    PROJECTION = dict(sec_per_epoch=per_epoch, images_per_epoch=real_imgs, epochs=REAL["epochs"], preload_s=pre, total_min=total_min)
    print("\n================ DRY-RUN TIMING (measured here, scaled to the full run) ================")
    print(f"backbone {BACKBONE} @ {IMG_SIZE}px | batch {BATCH_SIZE} | preload {'on' if PRELOAD else 'off'}")
    print(f"measured: {s_img * 1000:.1f} ms/train image, {v_img * 1000:.1f} ms/val image, checkpoint write {tt['ckpt_s']:.1f}s")
    print(f"peak GPU memory: {PEAK_GB:.1f} GB" if PEAK_GB else "peak GPU memory: n/a (no GPU)")
    print(f"full run = {real_imgs} train images/epoch x {REAL['epochs']} epochs -> about {per_epoch:.0f} s/epoch")
    print(f"projected full run: ~{total_min:.0f} min (image preload ~{pre / 60:.1f} min + training + final TTA evaluation)")
    print("not included: the one-time copy of images from Drive to Colab disk (a few minutes the first time).")
    print("note: the dry-run set is tiny, so fixed costs (data-loader start-up) inflate this a little; treat it as a slightly pessimistic estimate.")
    if total_min > 40:
        print(f"WARNING: projected {total_min:.0f} min > 40 min budget. Lower EPOCHS (e.g. 20) or BATCH_SIZE/IMG_SIZE before the real run.")
    print("=========================================================================================")


In [ ]:
import matplotlib.pyplot as plt
if len(hist_df) > 1:
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
    ax[0].plot(hist_df.epoch, hist_df.train_loss); ax[0].set_title("train loss (optimised, weighted)")
    ax[1].plot(hist_df.epoch, hist_df.val_loss); ax[1].set_title("val loss (unweighted CE)")
    ax[2].plot(hist_df.epoch, hist_df.val_macro_f1, label="macro-F1")
    ax[2].plot(hist_df.epoch, hist_df.val_mel_recall, label="mel recall"); ax[2].legend(); ax[2].set_title("val metrics")
    for a in ax: a.set_xlabel("epoch")
    plt.tight_layout(); plt.show()
else:
    print("Only one epoch (dry run): skipping curves.")

## 5 · Test evaluation (test set is used once, with the best-validation checkpoint)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

eval_net.load_state_dict(best_state)
val_logits, val_y = predict(val_loader, eval_net, tta=USE_TTA)
test_logits, test_y = predict(test_loader, eval_net, tta=USE_TTA)
VL, TL, vy, ty = val_logits.numpy(), test_logits.numpy(), val_y.numpy(), test_y.numpy()

bias, v_before, v_after = np.zeros(K), None, None
if POSTHOC:
    bias, v_before, v_after = tune_bias(VL, vy, MEL_RECALL_FLOOR)
raw_v, raw_t = summarize(VL, vy), summarize(TL, ty)
adj_v, adj_t = summarize(VL + bias, vy), summarize(TL + bias, ty)
final_t, final_logits = (adj_t, TL + bias) if POSTHOC else (raw_t, TL)

print(f"[{RUN_LABEL}] kept epoch {best_epoch} | TTA {USE_TTA} | EMA {USE_EMA}")
if POSTHOC:
    print("per-class offsets (", ", ".join(classes), "):", np.round(bias, 2))
    print(f"validation macro-F1 {v_before:.4f} -> {v_after:.4f} after tuning (this gain is on the data it was tuned on, so it is optimistic)\n")
tab = pd.DataFrame({
    "": ["raw", "post-hoc"] if POSTHOC else ["raw"],
    "val_F1": [raw_v["mf1"], adj_v["mf1"]][:2 if POSTHOC else 1],
    "TEST_F1": [raw_t["mf1"], adj_t["mf1"]][:2 if POSTHOC else 1],
    "TEST_bal_acc": [raw_t["bal"], adj_t["bal"]][:2 if POSTHOC else 1],
    "TEST_acc": [raw_t["acc"], adj_t["acc"]][:2 if POSTHOC else 1],
    "TEST_mel_rec": [raw_t["recalls"][mel_idx], adj_t["recalls"][mel_idx]][:2 if POSTHOC else 1],
    "TEST_AUC": [raw_t["auc"], adj_t["auc"]][:2 if POSTHOC else 1]})
print(tab.round(4).to_string(index=False))
print(f"\nFINAL (what is saved as the result): {'post-hoc' if POSTHOC else 'raw'}\n")
print(classification_report(ty, final_t["pred"], labels=list(range(K)), target_names=classes, digits=4, zero_division=0))

fig, ax = plt.subplots(1, 2, figsize=(16, 6.5))
cm = confusion_matrix(ty, final_t["pred"], labels=list(range(K)))
ConfusionMatrixDisplay(cm, display_labels=classes).plot(ax=ax[0], cmap="Blues", xticks_rotation=45, values_format="d", colorbar=False)
ax[0].set_title(f"Counts [{RUN_LABEL}]")
cmn = confusion_matrix(ty, final_t["pred"], labels=list(range(K)), normalize="true")
ConfusionMatrixDisplay(cmn, display_labels=classes).plot(ax=ax[1], cmap="Blues", xticks_rotation=45, values_format=".2f", colorbar=False)
ax[1].set_title("Row-normalised (diagonal = per-class recall)")
plt.tight_layout(); plt.show()

y_true, y_pred = ty, final_t["pred"]
acc, bal, mf1, auc, recalls = final_t["acc"], final_t["bal"], final_t["mf1"], final_t["auc"], final_t["recalls"]

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)

def block(s, with_cm=False):
    d = {"accuracy": float(s["acc"]), "balanced_accuracy": float(s["bal"]), "macro_f1": float(s["mf1"]),
         "macro_auc": float(s["auc"]), "recall": dict(zip(classes, [float(r) for r in s["recalls"]]))}
    if with_cm:
        d["confusion"] = confusion_matrix(ty, s["pred"], labels=list(range(K))).tolist()
    return d

results = {
    "tag": tag, "version": "v6", "weight_mode": WEIGHT_MODE, "seed": SEED, "dry_run": DRY_RUN, "best_epoch": int(best_epoch),
    "config": dict(backbone=BACKBONE, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, weight_decay=WEIGHT_DECAY, img_size=IMG_SIZE,
                   eff_beta=EFF_BETA, split_dir=SPLIT_DIR, schedule=USE_SCHEDULE, warmup_epochs=WARMUP_EPOCHS, ema=USE_EMA,
                   ema_decay=EMA_DECAY, label_smoothing=LABEL_SMOOTH, aug=AUG_LEVEL, grad_clip=GRAD_CLIP, select=SELECT,
                   tta=USE_TTA, posthoc=POSTHOC, mel_recall_floor=MEL_RECALL_FLOOR, member=MEMBER, mix_prob=MIX_PROB, mixup_alpha=MIXUP_ALPHA, cutmix_alpha=CUTMIX_ALPHA, color_constancy=COLOR_CONSTANCY, tta_mode=TTA_MODE, track=("reference_non_cnn" if BACKBONE in ("swin_t", "transnet") else "cnn"), lr_new=(LR_NEW if BACKBONE == "transnet" else None), metadata_used=False,
                   batch_size_requested=BATCH_SIZE_REQUESTED, preload=bool(PRELOAD), wd_excludes_norm=(BACKBONE == "convnext_tiny")),
    "class_weights": dict(zip(classes, [float(w) for w in class_weights])),
    "train_counts": dict(zip(classes, [int(c) for c in counts])),
    "per_epoch_targets": dict(zip(classes, [int(t) for t in targets])),
    "resample": dict(on=bool(RESAMPLE), mode=TARGET_MODE, cap=CAP_N, floor=FLOOR_N, max_repeat=MAX_REPEAT,
                     every_epoch=bool(RESAMPLE_EACH_EPOCH), lesion_aware=bool(LESION_AWARE)),
    "dedup": DEDUP_STATS,
    "test": block(final_t, True),                 # what the run is judged on (post-hoc if POSTHOC, else raw)
    "test_raw": block(raw_t),                     # same model, no post-hoc offsets
    "posthoc": {"on": bool(POSTHOC), "bias": dict(zip(classes, [float(b) for b in bias])),
                "val_f1_before": v_before, "val_f1_after": v_after},
    "best_val_macro_f1": float(best_f1),
    "history": history,
    "member": MEMBER,
    "timing": {"params_m": PARAMS_M, "sec_per_epoch": SEC_PER_EPOCH, "epochs": timing, "peak_gpu_gb": PEAK_GB, "probe_peak_gpu_gb": PROBE_PEAK_GB, "preload_s": PRELOAD_S,
               "training_min": run_min, "dry_run_projection": PROJECTION},
    "env": {"torch": torch.__version__, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"},
}
with open(f"{OUT_DIR}/results_{tag}.json", "w") as f:
    json.dump(results, f, indent=2)
softmax = lambda z: torch.softmax(torch.tensor(z), 1).numpy()
np.savez(f"{OUT_DIR}/probs_{tag}.npz", val_probs=softmax(VL), val_y=vy, test_probs=softmax(TL), test_y=ty)   # raw (no offsets)
if not DRY_RUN:
    torch.save({"model_state_dict": best_state, "class_to_idx": class_to_idx, "best_epoch": best_epoch, "backbone": BACKBONE,
                "img_size": IMG_SIZE, "bias": bias}, f"{OUT_DIR}/best_{tag}.pth")
if os.path.exists(CKPT):
    os.remove(CKPT)                 # results are safe now; the resume checkpoint is no longer needed
print("Saved results (and probabilities) to", OUT_DIR, "| tag:", tag)

## 6 · Compare everything saved so far
Reads saved files only (no training). `raw_f1` is the same model without post-hoc offsets (v4 runs only). Rows from earlier notebooks stay for reference; they used different protocols.
The second table is **mean ± std over seeds** for runs that differ only in `SEED`. That is the number to quote. With ~10 test images for `df` and 16 for `vasc`, one image moves those recalls by 10 and 6 points, so don't read much into single runs.

In [ ]:
rows = [{"run": "ORIGINAL baseline (no aug, val-loss select)", "acc": 0.8307, "bal_acc": 0.6504, "macro_f1": 0.6542,
         "auc": 0.9609, "mel_rec": 0.5625, "df_rec": 0.2000, "akiec_rec": 0.5122, "best_ep": 2, "raw_f1": np.nan}]
for f in sorted(glob.glob(f"{OUT_DIR}/results_*.json")):
    r = json.load(open(f))
    if r.get("dry_run"):
        continue
    t, raw = r["test"], r.get("test_raw")
    rows.append({"run": r["tag"], "acc": t["accuracy"], "bal_acc": t["balanced_accuracy"], "macro_f1": t["macro_f1"],
                 "auc": t["macro_auc"], "mel_rec": t["recall"]["mel"], "df_rec": t["recall"]["df"],
                 "akiec_rec": t["recall"]["akiec"], "best_ep": r["best_epoch"], "raw_f1": raw["macro_f1"] if raw else np.nan})
cmp_df = pd.DataFrame(rows)
pd.set_option("display.width", 250, "display.max_colwidth", 90)
print(cmp_df.round(4).to_string(index=False))

seeded = cmp_df[cmp_df["run"].str.contains(r"_seed\d+$")].copy()
seeded["config"] = seeded["run"].str.replace(r"_seed\d+$", "", regex=True)
agg = seeded.groupby("config").agg(n=("run", "count"), f1_mean=("macro_f1", "mean"), f1_std=("macro_f1", "std"),
                                   bal_mean=("bal_acc", "mean"), bal_std=("bal_acc", "std"),
                                   mel_mean=("mel_rec", "mean"), auc_mean=("auc", "mean"))
print("\nMean +- std over seeds (configs with 2+ seeds):")
print(agg[agg["n"] >= 2].round(4).to_string() if (agg["n"] >= 2).any() else "  (run the same config with 2+ different SEED values first)")

## 7 · Seed ensemble
Averages the saved probabilities of runs that differ only in `SEED` (v4 runs, 2+ seeds), then tunes the post-hoc offsets on the **averaged validation** probabilities and applies them to the averaged test probabilities. Ensembling several seeds usually gives a small but reliable gain, and it is the most dependable way to reduce single-run luck. Needs a session where the notebook has been run (it uses `tune_bias` from section 4).

In [ ]:
from collections import defaultdict
groups = defaultdict(list)
for f in sorted(glob.glob(f"{OUT_DIR}/probs_v[456]_*.npz")):
    tg = os.path.basename(f)[len("probs_"):-len(".npz")]
    if "DRYRUN" not in tg:
        groups[re.sub(r"_seed\d+$", "", tg)].append(f)

out = []
for cfg, files in groups.items():
    if len(files) < 2:
        continue
    zs = [np.load(f) for f in files]
    assert all((z["test_y"] == zs[0]["test_y"]).all() and (z["val_y"] == zs[0]["val_y"]).all() for z in zs), "different splits!"
    vp, tp = np.mean([z["val_probs"] for z in zs], 0), np.mean([z["test_probs"] for z in zs], 0)
    vy_, ty_ = zs[0]["val_y"], zs[0]["test_y"]
    VLe, TLe = np.log(vp + 1e-9), np.log(tp + 1e-9)
    b_, _, _ = tune_bias(VLe, vy_, MEL_RECALL_FLOOR)
    for name, lg in (("ensemble raw", TLe), ("ensemble + post-hoc", TLe + b_)):
        s = summarize(lg, ty_)
        out.append({"config": cfg[:70], "n_seeds": len(files), "variant": name, "macro_f1": s["mf1"], "bal_acc": s["bal"],
                    "mel_rec": s["recalls"][mel_idx], "auc": s["auc"]})
print(pd.DataFrame(out).round(4).to_string(index=False) if out else "Need 2+ non-dry runs of the same config with different SEEDs.")

## 8 · Reported by the paper vs measured here
Paper numbers are test **accuracy** (%). Measured numbers are the mean over all finished (non-dry) result files of that backbone found in `OUT_DIR`; `runs` says how many. Empty = no result file.

In [ ]:
paper = [("Swin Transformer", 99.70, "swin_t"), ("Inception-v3", 99.21, None), ("TransNet", 97.32, "transnet"),
         ("ViT", 91.46, None), ("VGG16", 69.91, None), ("ResNet-50", 60.38, "resnet50")]
res = {}
for f in sorted(glob.glob(f"{OUT_DIR}/results_*.json")):
    r = json.load(open(f))
    if not r.get("dry_run"):
        bb_ = r.get("config", {}).get("backbone")            # old v1-v3 result files have no backbone: skip them
        if bb_ and "test" in r:
            res.setdefault(bb_, []).append(r["test"])
rows = []
for name, acc, bb in paper:
    rs = res.get(bb, []) if bb else []
    rows.append({"model": name, "paper_acc_%": acc, "runs": len(rs) or "",
                 "measured_acc_%": round(100 * float(np.mean([x["accuracy"] for x in rs])), 2) if rs else "",
                 "measured_macro_F1": round(float(np.mean([x["macro_f1"] for x in rs])), 4) if rs else ""})
print(pd.DataFrame(rows).to_string(index=False))
print("\nNote: The paper uses pre-made per-class train/test folders and describes no lesion-grouped split. Ours is grouped by lesion with the original class distribution, so the numbers are not directly comparable.")